In [1]:
from google.colab import drive

drive.mount('/content/drive')


Mounted at /content/drive


In [22]:
import pandas as pd

# 엑셀 파일 불러오기 (경로를 본인의 파일 경로로 변경)
file_path = '/content/drive/MyDrive/리뷰데이터/review1.xlsx'
df = pd.read_excel(file_path)

# 데이터 확인
print(df.head())

   user_ID  color  totalscore
0        9  YOUTH           3
1      131  YOUTH           4
2      131  YOUTH           4
3      131  YOUTH           4
4      134  YOUTH           5


In [23]:
user_item_matrix = df.pivot_table(index='user_ID', columns='color', values='totalscore').fillna(0)

# 사용자-아이템 행렬 출력
print(user_item_matrix.head())

color     PINK GUAVA  01 피치 비치  02 스프링라이크 핑크  03 오리엔탈루비  04 젠틀 브릭  05 핑크 미터  \
user_ID                                                                       
1                0.0       0.0           0.0        0.0       0.0       0.0   
2                0.0       0.0           0.0        0.0       0.0       0.0   
3                0.0       0.0           0.0        0.0       0.0       0.0   
4                0.0       0.0           0.0        0.0       0.0       0.0   
5                0.0       0.0           0.0        0.0       0.0       0.0   

color    06 핑크 하이웨이  07 베리 피즈  08 어텀 뮬리  09 레드체크아웃  ...  TANGERINE TANGO  \
user_ID                                             ...                    
1               0.0       0.0       0.0        0.0  ...              0.0   
2               0.0       0.0       0.0        0.0  ...              0.0   
3               0.0       0.0       0.0        0.0  ...              0.0   
4               0.0       0.0       0.0        0.0  ...           

In [24]:
from sklearn.decomposition import TruncatedSVD

# SVD 모델 생성 (n_components는 잠재 요인의 개수를 설정)
svd = TruncatedSVD(n_components=10)

# 사용자-아이템 행렬에 SVD 적용
latent_matrix = svd.fit_transform(user_item_matrix)

# 잠재 요인 행렬 확인
print(latent_matrix.shape)  # 사용자별 잠재 요인
print(latent_matrix)

(798, 10)
[[ 1.78521286e-07 -8.04000700e-06  1.48069576e-05 ... -3.65252121e-03
   2.91479792e-04 -2.24177676e-03]
 [ 6.37863721e-04  4.97812667e+00 -1.92158803e-02 ... -9.33165921e-02
   3.16824348e-02  7.52519200e-02]
 [ 1.24000162e-07  4.12714435e-05  5.16181996e-04 ...  1.07606056e-02
   6.89155360e-03  4.52594700e-03]
 ...
 [ 3.77218783e-05  1.72402664e-02  1.49449727e-01 ... -1.91296653e+00
   1.91298086e+00 -1.24460072e-01]
 [ 7.13133234e-09  1.30388568e-06 -2.45445106e-07 ... -3.66796861e-04
  -1.06631876e-04  3.45249797e-03]
 [ 3.74781554e-04  1.53783335e-01  2.38758850e-03 ...  2.31748932e+00
   1.26024416e+00 -1.20757237e+00]]


In [10]:
!pip install pyspark

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.3/317.3 MB 3.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pyspark: filename=pyspark-3.5.3-py2.py3-none-any.whl size=317840625 sha256=4c2771444787a5b8415ce4aeeb27fe7eb1543d57b1137927ab632f86ce16c37d
  Stored in directory: /root/.cache/pip/wheels/1b/3a/92/28b93e2fbfdbb07509ca4d6f50c5e407f48dce4ddbda69a4ab
Successfully built pyspark


In [25]:
print(df.dtypes)

# 특정 컬럼의 타입이 문자열이라면, 숫자로 변환
df['color'] = pd.to_numeric(df['color'], errors='coerce')

user_ID        int64
color         object
totalscore     int64
dtype: object


In [26]:
from sklearn.preprocessing import LabelEncoder

# 레이블 인코더 생성
label_encoder = LabelEncoder()

# color 열 인코딩
df['color'] = label_encoder.fit_transform(df['color'])

# 결과 확인
print(df['color'].head())
print(df.dtypes)  # 데이터 타입 확인

0    0
1    0
2    0
3    0
4    0
Name: color, dtype: int64
user_ID       int64
color         int64
totalscore    int64
dtype: object


In [18]:
try:
    df = pd.get_dummies(df, columns=['color'], drop_first=True)
except Exception as e:
    print("Error during one-hot encoding:", e)

Error during one-hot encoding: "None of [Index(['color'], dtype='object')] are in the [columns]"


In [27]:
from pyspark.ml.recommendation import ALS
from pyspark.sql import SparkSession
import pandas as pd

# Spark 세션 생성
spark = SparkSession.builder.appName('ALSExample').getOrCreate()

# pandas DataFrame을 Spark DataFrame으로 변환
df_spark = spark.createDataFrame(df)

# ALS 모델 설정 및 학습
als = ALS(maxIter=10, regParam=0.1, userCol="user_ID", itemCol="color", ratingCol="totalscore", coldStartStrategy="drop")
model = als.fit(df_spark)

# 예측 생성 (학습 데이터에 대한 예측)
predictions = model.transform(df_spark)
predictions.show()

+-------+-----+----------+----------+
|user_ID|color|totalscore|prediction|
+-------+-----+----------+----------+
|    148|    0|         4| 3.8469746|
|    496|    0|         5|  4.808718|
|    148|    0|         4| 3.8469746|
|    463|    0|         5| 4.3278465|
|    463|    0|         4| 4.3278465|
|    463|    0|         4| 4.3278465|
|    463|    0|         5| 4.3278465|
|    392|    0|         4|  4.568282|
|    623|    0|         3|  2.885231|
|    623|    0|         3|  2.885231|
|     31|    0|         2| 1.9234873|
|    580|    0|         5|  4.808718|
|     53|    0|         5|  4.808718|
|    255|    0|         4|  4.488137|
|    513|    0|         4| 3.8469746|
|     78|    0|         4| 3.8469746|
|    613|    0|         5|  4.808718|
|    673|    0|         5|  4.808718|
|    362|    0|         5|  4.808718|
|    108|    0|         5|  4.808718|
+-------+-----+----------+----------+
only showing top 20 rows



In [28]:
# 특정 사용자에게 추천할 제품 5개 추출
user_id = 123  # 추천을 원하는 사용자 ID

# 사용자에게 추천할 제품 예측
user_recommendations = model.recommendForAllUsers(5)  # 각 사용자에게 5개의 제품 추천
user_recommendations.show()

+-------+----------------+
|user_ID| recommendations|
+-------+----------------+
|      1| [{0, 4.808718}]|
|      2| [{0, 4.808718}]|
|      3| [{0, 4.808718}]|
|      4|[{0, 3.8469746}]|
|      5| [{0, 4.808718}]|
|      6| [{0, 4.808718}]|
|      7|[{0, 3.8469746}]|
|      8|[{0, 3.8469746}]|
|      9|[{0, 2.8852308}]|
|     10| [{0, 4.808718}]|
|     11| [{0, 4.808718}]|
|     12| [{0, 4.808718}]|
|     13| [{0, 4.808718}]|
|     14|[{0, 3.1256669}]|
|     15| [{0, 4.808718}]|
|     16|[{0, 3.8469746}]|
|     17| [{0, 4.808718}]|
|     18|[{0, 3.8469746}]|
|     19| [{0, 4.808718}]|
|     20| [{0, 4.808718}]|
+-------+----------------+
only showing top 20 rows



In [32]:
print(df.head())

   user_ID  color  totalscore
0        9      0           3
1      131      0           4
2      131      0           4
3      131      0           4
4      134      0           5
